# Notebook 04 - Manual Experiment Tracking Before MLflow

This notebook shows the pain of manual experiment tracking.

Students will understand why MLflow is needed.

In [1]:
# Core libraries used throughout the Day 1 notebooks
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# Display settings make notebook output easier to read during training
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 120)

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import time

In [2]:
churn_df = pd.read_csv('../datasets/customer_churn_training.csv')
X = churn_df.drop(columns=['customer_id', 'churn'])
y = churn_df['churn']

numerical_features = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = X.select_dtypes(include=['object']).columns.tolist()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

numeric_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_pipeline, numerical_features),
    ('cat', categorical_pipeline, categorical_features)
])

## Manual experiment loop

We train Random Forest with different parameters.

In real life, after many runs, it becomes difficult to remember:

- Which parameters were used
- Which metrics were achieved
- Which run produced the best model
- Which dataset version was used
- Which notebook/code version produced the result

In [3]:
experiment_results = []

# Different hyperparameter combinations to try
parameter_grid = [
    {'n_estimators': 50,  'max_depth': 4},
    {'n_estimators': 100, 'max_depth': 4},
    {'n_estimators': 100, 'max_depth': 6},
    {'n_estimators': 150, 'max_depth': 8},
    {'n_estimators': 200, 'max_depth': None},
]

for run_number, params in enumerate(parameter_grid, start=1):
    start_time = time.time()

    model_pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('model', RandomForestClassifier(
            n_estimators=params['n_estimators'],
            max_depth=params['max_depth'],
            random_state=42
        ))
    ])

    model_pipeline.fit(X_train, y_train)
    y_pred = model_pipeline.predict(X_test)

    run_result = {
        'run_number': run_number,
        'algorithm': 'RandomForestClassifier',
        'n_estimators': params['n_estimators'],
        'max_depth': params['max_depth'],
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred, zero_division=0),
        'recall': recall_score(y_test, y_pred, zero_division=0),
        'f1_score': f1_score(y_test, y_pred, zero_division=0),
        'duration_seconds': round(time.time() - start_time, 3)
    }

    experiment_results.append(run_result)

manual_tracking_df = pd.DataFrame(experiment_results)
manual_tracking_df

,run_number,algorithm,n_estimators,max_depth,accuracy,precision,recall,f1_score,duration_seconds
0,1,RandomForestClassifier,50,4.0,0.858333,0.0,0.000000,0.000000,0.041
1,2,RandomForestClassifier,100,4.0,0.850000,0.0,0.000000,0.000000,0.064
2,3,RandomForestClassifier,100,6.0,0.850000,0.0,0.000000,0.000000,0.067
3,4,RandomForestClassifier,150,8.0,0.841667,0.0,0.000000,0.000000,0.102
4,5,RandomForestClassifier,200,NaN,0.858333,0.5,0.117647,0.190476,0.138


In [4]:
# Identify best experiment manually
best_run = manual_tracking_df.sort_values(by='f1_score', ascending=False).iloc[0]
best_run

run_number                               5
algorithm           RandomForestClassifier
n_estimators                           200
max_depth                              NaN
accuracy                          0.858333
precision                              0.5
recall                            0.117647
f1_score                          0.190476
duration_seconds                     0.138
Name: 4, dtype: object

In [5]:
# Save the manual tracking table.
# This is similar to what MLflow Tracking UI gives automatically.
manual_tracking_df.to_csv('../outputs/manual_experiment_tracking.csv', index=False)
print('Saved manual tracking results to ../outputs/manual_experiment_tracking.csv')

Saved manual tracking results to ../outputs/manual_experiment_tracking.csv


## Why this is not enough

Manual CSV tracking is okay for 5 runs, but not for real projects.

Problems:

1. The CSV may be overwritten accidentally.
2. Model files are not connected to metrics.
3. Plots and artifacts are not organized.
4. Dataset/code version is not tracked properly.
5. Team members cannot easily compare experiments.
6. Deployment workflow is missing.

This is the exact problem MLflow solves.